# 01 - Data Exploration
## Polymarket Obfuscation Detection Pipeline

This notebook explores the trade data ingested from Polymarket for the Iran-Israel nuclear strike market.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from src.storage.postgres_store import PostgresStore
from src.models.database import Wallet, Trade, Market

## Connect to Database

In [ ]:
store = PostgresStore()
print("Connected to PostgreSQL database")

## Load Market Information

In [ ]:
market = store.session.query(Market).first()
if market:
    print(f"Market Question: {market.question}")
    print(f"Condition ID: {market.condition_id}")
    print(f"Slug: {market.slug}")
    print(f"Tokens: {market.tokens}")
    print(f"Closed: {bool(market.closed)}")
else:
    print("No market data found. Run the ingestion pipeline first.")

## Load All Trades

In [ ]:
trades = store.session.query(Trade).all()
print(f"Total trades loaded: {len(trades)}")

In [ ]:
df_trades = pd.DataFrame([
    {
        'tx_hash': t.tx_hash,
        'block_number': t.block_number,
        'timestamp': t.timestamp,
        'market_id': t.market_id,
        'wallet_address': t.wallet_address,
        'side': t.side,
        'amount_usd': t.amount_usd,
        'maker': t.maker,
        'taker': t.taker
    }
    for t in trades
])

df_trades['timestamp'] = pd.to_datetime(df_trades['timestamp'])
df_trades.head()

## Basic Statistics

In [ ]:
print("=== Trade Statistics ===")
print(f"Total Volume: ${df_trades['amount_usd'].sum():,.2f}")
print(f"Average Trade Size: ${df_trades['amount_usd'].mean():,.2f}")
print(f"Median Trade Size: ${df_trades['amount_usd'].median():,.2f}")
print(f"Max Trade Size: ${df_trades['amount_usd'].max():,.2f}")
print(f"Min Trade Size: ${df_trades['amount_usd'].min():,.2f}")

In [ ]:
print("=== Side Distribution ===")
print(df_trades['side'].value_counts())

## Volume Over Time

In [ ]:
df_trades.set_index('timestamp', inplace=True)
df_trades.resample('D')['amount_usd'].sum().plot(figsize=(12, 6))
plt.title('Daily Trading Volume')
plt.xlabel('Date')
plt.ylabel('Volume (USD)')
plt.grid(True)
plt.tight_layout()
plt.show()

## Top Traders

In [ ]:
top_traders = df_trades.groupby('wallet_address').agg({
    'amount_usd': ['sum', 'count', 'mean'],
    'side': lambda x: (x == 'BUY').sum()
}).round(2)

top_traders.columns = ['total_volume', 'trade_count', 'avg_volume', 'buy_count']
top_traders = top_traders.sort_values('total_volume', ascending=False)
top_traders.head(20)

## Trade Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_trades['amount_usd'].hist(bins=50, ax=axes[0])
axes[0].set_title('Trade Size Distribution')
axes[0].set_xlabel('Amount (USD)')
axes[0].set_ylabel('Frequency')

df_trades[df_trades['amount_usd'] < 1000]['amount_usd'].hist(bins=50, ax=axes[1])
axes[1].set_title('Trade Size Distribution (< $1000)')
axes[1].set_xlabel('Amount (USD)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Unique Wallets Analysis

In [ ]:
print(f"Unique Wallets: {df_trades['wallet_address'].nunique()}")
print(f"Total Trades: {len(df_trades)}")
print(f"Trades per Wallet: {len(df_trades) / df_trades['wallet_address'].nunique():.2f}")

## Hour of Day Analysis

In [ ]:
df_trades['hour'] = df_trades.index.hour
df_trades.groupby('hour')['amount_usd'].sum().plot(kind='bar', figsize=(12, 5))
plt.title('Volume by Hour of Day')
plt.xlabel('Hour (UTC)')
plt.ylabel('Volume (USD)')
plt.xticks(rotation=0)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

## Wallet Distribution by Volume

In [ ]:
volume_by_wallet = df_trades.groupby('wallet_address')['amount_usd'].sum().sort_values(ascending=False)

top_10_pct = volume_by_wallet.head(10).sum() / volume_by_wallet.sum() * 100
print(f"Top 10 wallets account for {top_10_pct:.1f}% of total volume")

fig, ax = plt.subplots(figsize=(10, 6))
volume_by_wallet.head(20).plot(kind='bar', ax=ax)
ax.set_title('Top 20 Wallets by Volume')
ax.set_xlabel('Wallet Address')
ax.set_ylabel('Total Volume (USD)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
store.close()
print("Database connection closed")